<a href="https://colab.research.google.com/github/rubyratcha-19/ge338_lab2/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# STEP 0 — INIT
# ===============================
import ee
import geemap

ee.Initialize(project='inlaid-reactor-457503-u6')

Map = geemap.Map()

# ===============================
# STEP 1 — AOI (นนทบุรี)
# ===============================
th = ee.FeatureCollection("FAO/GAUL/2015/level1")

roi = th.filter(ee.Filter.eq('ADM1_NAME', 'Nonthaburi')).geometry()

Map.centerObject(roi, 10)
Map.addLayer(roi, {'color':'red'}, 'Nonthaburi')

# ===============================
# STEP 2 — SENTINEL-2 (CLOUD MASK + SCALE)
# ===============================
def maskS2(image):
    qa = image.select('QA60')

    cloud = 1 << 10
    cirrus = 1 << 11

    mask = qa.bitwiseAnd(cloud).eq(0).And(
           qa.bitwiseAnd(cirrus).eq(0))

    return image.updateMask(mask).divide(10000)

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR')
          .filterBounds(roi)
          .filterDate('2020-01-01', '2020-12-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .map(maskS2))

# ===============================
# STEP 3 — COMPOSITE
# ===============================
s2 = s2_col.median().clip(roi)

# ===============================
# STEP 4 — RGB (อันนี้สำคัญ มึงต้องขึ้น)
# ===============================
rgb_vis = {
    'bands': ['B4','B3','B2'],
    'min': 0.05,
    'max': 0.3,
    'gamma': 1.2
}

Map.addLayer(s2, rgb_vis, 'RGB')

# ===============================
# STEP 5 — NDVI
# ===============================
ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')

ndvi_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white','lightgreen','green','darkgreen']
}

Map.addLayer(ndvi, ndvi_vis, 'NDVI')

# ===============================
# STEP 6 — NDWI (Water)
# ===============================
ndwi = s2.normalizedDifference(['B3','B8']).rename('NDWI')

ndwi_vis = {
    'min': -1,
    'max': 1,
    'palette': ['brown','white','blue']
}

Map.addLayer(ndwi, ndwi_vis, 'NDWI')

# ===============================
# STEP 7 — NDBI (Built-up)
# ===============================
ndbi = s2.normalizedDifference(['B11','B8']).rename('NDBI')

ndbi_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['white','gray','black']
}

Map.addLayer(ndbi, ndbi_vis, 'NDBI')

# ===============================
# SHOW MAP
# ===============================
Map

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [ ]:
# ==============================
# STEP 0 — INSTALL + AUTH
# ==============================
!pip install earthengine-api geemap matplotlib

import ee
import geemap
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='inlaid-reactor-457503-u6')

Map = geemap.Map()

# ==============================
# STEP 1 — AOI (นนทบุรี)
# ==============================
thai = ee.FeatureCollection('FAO/GAUL/2015/level1')
roi = thai.filter(ee.Filter.eq('ADM1_NAME', 'Nonthaburi'))

Map.centerObject(roi, 10)
Map.addLayer(roi, {}, 'ROI')

# ==============================
# STEP 2 — Sentinel-2 (FIX BAND ERROR)
# ==============================
def maskS2(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
    return (image.updateMask(mask)
            .select(['B2','B3','B4','B8','B11'])
            .copyProperties(image, image.propertyNames()))

s2 = (ee.ImageCollection('COPERNICUS/S2_SR')
      .filterBounds(roi)
      .filterDate('2020-01-01', '2020-12-31')
      .map(maskS2)
      .median()
      .clip(roi))

Map.addLayer(s2, {'bands':['B4','B3','B2'], 'min':0, 'max':3000}, 'RGB')

# ==============================
# STEP 3 — INDICES
# ==============================
ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi = s2.normalizedDifference(['B11','B8']).rename('NDBI')

Map.addLayer(ndvi, {'min':0,'max':1,'palette':['white','green']}, 'NDVI')
Map.addLayer(ndwi, {'min':-1,'max':1,'palette':['brown','blue']}, 'NDWI')
Map.addLayer(ndbi, {'min':-1,'max':1,'palette':['white','red']}, 'NDBI')

# ==============================
# STEP 4 — LST
# ==============================
lst = (ee.ImageCollection('MODIS/006/MOD11A2')
       .filterBounds(roi)
       .filterDate('2020-01-01','2020-12-31')
       .select('LST_Day_1km')
       .mean()
       .multiply(0.02)
       .subtract(273.15)
       .rename('LST')
       .clip(roi))

Map.addLayer(lst, {'min':25,'max':45,'palette':['blue','yellow','red']}, 'LST')

# ==============================
# STEP 5 — AIR TEMP
# ==============================
air = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
       .select('temperature_2m')
       .filterDate('2020-01-01','2020-12-31')
       .mean()
       .subtract(273.15)
       .rename('Air')
       .clip(roi))

Map.addLayer(air, {'min':20,'max':35,'palette':['blue','green','red']}, 'Air Temp')

# ==============================
# STEP 6 — NORMALIZE
# ==============================
def normalize(img, name):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=1000,
        maxPixels=1e13
    )
    minv = ee.Number(stats.get(name + '_min'))
    maxv = ee.Number(stats.get(name + '_max'))
    return img.subtract(minv).divide(maxv.subtract(minv)).rename(name)

ndvi_n = normalize(ndvi, 'NDVI')
ndwi_n = normalize(ndwi, 'NDWI')
ndbi_n = normalize(ndbi, 'NDBI')
lst_n  = normalize(lst, 'LST')
air_n  = normalize(air, 'Air')

# ==============================
# STEP 7 — MODEL (UHS)
# ==============================
UHS = (lst_n.multiply(0.4)
       .add(air_n.multiply(0.2))
       .add(ndbi_n.multiply(0.2))
       .subtract(ndvi_n.multiply(0.15))
       .subtract(ndwi_n.multiply(0.05))
       .rename('UHS'))

Map.addLayer(UHS, {'min':0,'max':1,'palette':['green','yellow','red']}, 'UHS')

# ==============================
# STEP 8 — VALIDATION (UHII DATASET จริง)
# ==============================
uhii = (ee.ImageCollection('projects/sat-io/open-datasets/UHII/SAT')
        .filterDate('2020-01-01','2020-12-31')
        .mean()
        .multiply(0.01)
        .rename('UHII')
        .clip(roi))

Map.addLayer(uhii, {'min':0,'max':2,'palette':['blue','white','red']}, 'UHII')

# ==============================
# STEP 9 — CORRELATION (สำคัญมาก)
# ==============================
combined = UHS.addBands(uhii)

corr = combined.reduceRegion(
    reducer=ee.Reducer.pearsonsCorrelation(),
    geometry=roi,
    scale=1000,
    maxPixels=1e13
)

print("Correlation UHS vs UHII:", corr.getInfo())

# ==============================
# STEP 10 — HISTOGRAM
# ==============================
hist = UHS.reduceRegion(
    reducer=ee.Reducer.histogram(50),
    geometry=roi,
    scale=500,
    maxPixels=1e13
).getInfo()

bins = hist['UHS']['bucketMeans']
counts = hist['UHS']['histogram']

plt.figure()
plt.bar(bins, counts)
plt.title('UHS Distribution')
plt.xlabel('UHS')
plt.ylabel('Frequency')
plt.show()

# ==============================
# STEP 11 — SENSITIVITY ANALYSIS
# ==============================
UHS_high = (lst_n.multiply(0.48)
            .add(air_n.multiply(0.2))
            .add(ndbi_n.multiply(0.2))
            .subtract(ndvi_n.multiply(0.15))
            .subtract(ndwi_n.multiply(0.05)))

UHS_low = (lst_n.multiply(0.32)
           .add(air_n.multiply(0.2))
           .add(ndbi_n.multiply(0.2))
           .subtract(ndvi_n.multiply(0.15))
           .subtract(ndwi_n.multiply(0.05)))

diff = UHS_high.subtract(UHS_low)

Map.addLayer(diff, {
    'min':-0.2,'max':0.2,
    'palette':['blue','white','red']
}, 'Sensitivity')

# ==============================
# FINAL
# ==============================
Map